# Benchmarking ECDLP functions

In [1]:
import qrisp
import numpy as np
import src.classical.ec_arithmetic as clECarithm
import src.quantum.ec_arithmetic as qECarithm

In [2]:
def t_depth_indicator(op):
    if op.name in ["t", "t_dg"]:
        return 1
    else:
        return 0

def benchmark_circuit(res):
    
    compiled_circuit = res.qs.compile(compile_mcm = True, gate_speed = t_depth_indicator, workspace = res.size)
    qubit_count = compiled_circuit.num_qubits()
    gate_counts = compiled_circuit.transpile().count_ops()
    #print(gate_counts)
    cnot_count = gate_counts.get('cx', 0) + 0.5*gate_counts.get('c_if_cz', 0)
    t_count = gate_counts.get('t', 0) + gate_counts.get('t_dg', 0)
    t_depth = compiled_circuit.depth(depth_indicator = t_depth_indicator)  # Assumes depth of T gates aligns with circuit depth
    return {
        "qubit_count": qubit_count,
        "t_count": t_count,
        "cnot_count": cnot_count,
        "t_depth": t_depth
    }

In [3]:
p=7
a=5
b=4
curve = clECarithm.EllCurve(a, b, p)

In [4]:
benchmark_results_kaliski = {}
for val in range(1,p):
    v = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
    v[:] = val
    m = qrisp.QuantumArray(qtype=qrisp.QuantumBool(), shape=(2 * p.bit_length(),))

    res_kaliski = qECarithm.kaliski_quantum(v, p, m)
    for a in m:
        a.delete()
    benchmark_results_kaliski[val] = benchmark_circuit(res_kaliski)


In [5]:
# Display results in a readable format
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_kaliski, orient="index")
results_df.index.name = "Value of v"
print(results_df)

            qubit_count  t_count  cnot_count  t_depth
Value of v                                           
1                    30     2918      5651.0      855
2                    30     2918      5651.0      855
3                    30     2918      5651.0      855
4                    30     2918      5651.0      855
5                    30     2918      5651.0      855
6                    30     2918      5651.0      855


In [6]:
anc_values = [
    (2, 6),
    # (4, 2),
    # (0, 5),
    # (5, 0),
]

mod_p = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
G = [3, 2]

benchmark_results_ec_add = {}

for anc_pair in anc_values:
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = list(anc_pair)
    res_ec_add = qECarithm.qrisp_ell_add_inpl(anc, G, p)
    bm = benchmark_circuit(res_ec_add[0])
    benchmark_results_ec_add[tuple(anc_pair)] = bm

In [7]:
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_ec_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
print(results_df)

     qubit_count  t_count  cnot_count  t_depth
2 6           64    16388     37331.5     3829
4 2           64    16388     37331.5     3829
0 5           64    16388     37331.5     3830
5 0           64    16388     37331.5     3838


In [8]:
x1 = qrisp.QuantumModulus(2**(p.bit_length()))
x1[:] = 1
mod_p = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
G = [3, 2]

benchmark_results_ec_ctrl_add = {}

for anc_pair in anc_values:
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = list(anc_pair)
    res_ec_ctrl_add = qECarithm.qrisp_ell_mult_add(anc, G, x1, p)
    bm = benchmark_circuit(res_ec_ctrl_add[0])
    benchmark_results_ec_ctrl_add[tuple(anc_pair)] = bm

In [9]:
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_ec_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
print(results_df)

     qubit_count  t_count  cnot_count  t_depth
2 6           64    16388     37331.5     3829
4 2           64    16388     37331.5     3829
0 5           64    16388     37331.5     3830
5 0           64    16388     37331.5     3838


In [5]:
# Define the values of ancilla to test
anc_values = [
    (2, 6),
    # (4, 2),
    # (0, 5),
    # (5, 0),
]


# Define the elliptic curve parameters and constants
curve = clECarithm.EllCurve(a, b, p)
mod_p = qrisp.QuantumModulus(p)
G = [3, 2]
P = [0, 2]

benchmark_results_mult_add = {}

for anc_pair in anc_values:
    print(anc_pair)
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = list(anc_pair)
    x1 = qrisp.QuantumModulus(2**(p.bit_length()))
    x2 = qrisp.QuantumModulus(2**(p.bit_length()))

    qrisp.h(x1)
    qrisp.h(x2)
    anc = qECarithm.qrisp_ell_mult_add(G, anc, x1, curve)
    anc = qECarithm.qrisp_ell_mult_add(P, anc, x2, curve)
    qrisp.QFT(x1, inv=True)
    qrisp.QFT(x2, inv=True)
    print(x2.qs.qv_list)

    bm = benchmark_circuit(anc[0])

    # Store results in the dictionary
    benchmark_results_mult_add[tuple(anc_pair)] = bm


(2, 6)
[<QuantumModulus 'qf_49'>, <QuantumModulus 'qf_49_1'>, <QuantumModulus 'qf_50'>, <QuantumModulus 'qf_51'>]
(4, 2)
[<QuantumModulus 'qf_49'>, <QuantumModulus 'qf_49_1'>, <QuantumModulus 'qf_66'>, <QuantumModulus 'qf_67'>]
(0, 5)
[<QuantumModulus 'qf_49'>, <QuantumModulus 'qf_49_1'>, <QuantumModulus 'qf_82'>, <QuantumModulus 'qf_83'>]
(5, 0)
[<QuantumModulus 'qf_49'>, <QuantumModulus 'qf_49_1'>, <QuantumModulus 'qf_98'>, <QuantumModulus 'qf_99'>]


In [14]:
# Display results in a readable format
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_mult_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
results_df.columns.name = "Benchmark Metrics"
print(results_df)

Benchmark Metrics  qubit_count  t_count  cnot_count  t_depth
3 2                         42    85842    280200.0    37262
4 2                         48   171684    560400.0    74505
0 5                         54   257526    840588.0   111726
5 0                         60   343368   1120776.0   148917
0 2                         66   429210   1400964.0   186029
4 5                         72   515052   1681164.0   204642
2 1                         78   600894   1961364.0   223272
3 5                         84   686736   2241564.0   241873


In [ ]:
# benchmark_results = {}

# benchmark_results["kaliski_quantum"] = benchmark_circuit(res_kaliski)
# benchmark_results["ec_addition"] = benchmark_circuit(res_ec_add[0])
# benchmark_results["controlled_addition"] = benchmark_circuit(res_ec_ctrl_add[0])
# benchmark_results["Shor's algorithm"] = benchmark_circuit(anc[0])

In [ ]:
# import pandas as pd

# # Display results in a tabular format
# results_df = pd.DataFrame(benchmark_results).T
# print(results_df)